# LowLight GAN - training on Colab
**Runtime -> Change runtime type -> T4 GPU**, then run the cells in order.

Before starting: upload `lowlight-gan.zip` (this project) and your image dataset zip (e.g. COCO val2017) to Google Drive.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT these two paths to match your Drive
PROJECT_ZIP = '/content/drive/MyDrive/lowlight-gan.zip'
DATA_ZIP    = '/content/drive/MyDrive/coco_val2017.zip'

!unzip -q -o $PROJECT_ZIP -d /content
%cd /content/lowlight-gan
!pip -q install -r requirements.txt
!mkdir -p raw && unzip -q -o $DATA_ZIP -d raw
!python prepare_data.py --src raw --dst data/clean --max_side 640

# Hold out 100 images for testing BEFORE training (no leakage)
!mkdir -p data/test_clean && ls data/clean | tail -n 100 | xargs -I{} mv data/clean/{} data/test_clean/

## 1. Train the GAN (main model)
~1-2 min/epoch on a T4 for ~5k images. Checkpoints & sample grids are written every epoch, so you can stop/resume safely.

In [ ]:
!python train.py --data data/clean --name gan --epochs 60 --batch 16

## 2. Ablation: same U-Net, no discriminator (L1 only)
This answers *"does the GAN actually help?"* - a question examiners love.

In [ ]:
!python train.py --data data/clean --name unet_l1 --lambda_gan 0 --epochs 60 --batch 16

## 3. Evaluate vs. baselines
The 100 images moved to `data/test_clean` earlier were never seen in training. Optionally also test on the real LOL dataset (`data/LOL/eval15`).

In [ ]:
!python evaluate.py --data data/test_clean --ckpt GAN=checkpoints/gan/generator_best.pt --ckpt "U-Net (L1 only)=checkpoints/unet_l1/generator_best.pt"

In [ ]:
# Real-world check (optional, needs LOL dataset)
# !python evaluate.py --lol data/LOL/eval15 --ckpt GAN=checkpoints/gan/generator_best.pt --ckpt "U-Net (L1 only)=checkpoints/unet_l1/generator_best.pt"

In [ ]:
from IPython.display import Image, display
display(Image('checkpoints/gan/curves.png')); display(Image('results/comparison_synthetic.png'))

## 4. Save everything back to Drive, then run the UI on your laptop
`streamlit run app.py` (copy `checkpoints/*/generator_best.pt` and `results/` into the project folder).

In [ ]:
!mkdir -p /content/drive/MyDrive/lowlight_outputs
!cp -r checkpoints results /content/drive/MyDrive/lowlight_outputs/